In [10]:
from mlx_lm import load, generate

model, tokenizer = load("mlx-community/Qwen2.5-0.5B-Instruct-4bit")

prompt = "Write a story about Einstein"
messages = [{"role": "user", "content": prompt}]
prompt = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
)
text = generate(model, tokenizer, prompt=prompt, verbose=True)

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Einstein was a brilliant and innovative mind who was known for his groundbreaking discoveries and contributions to the field of science and mathematics. He was a true polymath, with a wide range of skills and interests that allowed him to explore different areas of knowledge and challenge the boundaries of what was thought to be possible.

Einstein's most famous discovery was the theory of relativity, which explained the behavior of objects in motion and the nature of time and space. His work in this area was groundbreaking and had a profound impact on the world around us. He also made significant contributions to the field of quantum mechanics, which was a revolutionary development that challenged the traditional understanding of the world.

Einstein was also known for his ability to think outside the box and come up with creative solutions to problems. He was a true visionary and a true innovator, and his work continues to inspire and challenge us to think differently about the world

In [15]:
import pandas as pd
df = pd.read_csv("results.csv")

def format_example(row):
    return {
        "text": (
            f"<s>[INST] {row['question']} [/INST] "
            f"{row['agent1_reasoning']}\n\nAnswer: {row['true_answer']}</s>"
        )
    }
df

,year,section,question_id,question,true_answer,agent1_answer,agent1_reasoning,agent2_verdict,agent2_justification,correct_flag
0,2011,freshman_sophomore_test,question_1,A convex polygon has 19 sides. Find the sum of...,3060,3060,"We are asked: ""A convex polygon has 19 sides. ...",CORRECT,The student's answer matches the correct answe...,1
1,2011,freshman_sophomore_test,question_2,"If (x^2 - y^2) = 0 and x + y = 17.534, find th...",0,0,We are given: (x^2 - y^2) = 0 and x + y = 17.5...,CORRECT,The equation x^2 - y^2 = 0 implies (x-y)(x+y)=...,1
2,2011,freshman_sophomore_test,question_3,Two circles are constructed such that one has ...,16,16,"We are given: ""Two circles are constructed suc...",CORRECT,The student's answer matches the correct numer...,1
3,2011,freshman_sophomore_test,question_4,"Casey invested part of $100,000 in bonds that ...",68000,68000,"We are given: Casey invested part of $100,000 ...",CORRECT,The student's answer matches the correct numer...,1
4,2011,freshman_sophomore_test,question_6,A bag of shapes contains the following polygon...,2/3,2/3,"We are asked: ""A bag of shapes contains the fo...",CORRECT,The student's answer matches the correct simpl...,1
...,...,...,...,...,...,...,...,...,...,...
666,2025,two_person_speed_event,question_8,Tennis balls come only in boxes of 7 and 16. T...,140,140,"We are given: ""Tennis balls come only in boxes...",CORRECT,The student's answer matches the correct numer...,1
667,2025,two_person_speed_event,question_9,The lengths of all sides of a right triangle a...,84,84,"We are given: ""The lengths of all sides of a r...",CORRECT,The student's answer matches the correct answe...,1
668,2025,two_person_speed_event,question_10,A coin flip will be used before a baseball gam...,1/2,1/2,"We are given: ""A coin flip will be used before...",CORRECT,The probability of calling correctly is indepe...,1
669,2025,two_person_speed_event,question_11,"A sum of $5,000 is invested and grows to $24,0...",16,16,"We are given: ""A sum of $5,000 is invested and...",CORRECT,The student's answer matches the correct round...,1


In [16]:
df = df[df["agent2_verdict"] == "CORRECT"]
print(f"Kept {len(df)} examples")

# Split train/valid
train = df.sample(frac=0.9, random_state=42)
valid = df.drop(train.index)

Kept 594 examples


In [27]:
import json
with open("train.jsonl", "w") as f:
    for _, row in train.iterrows():
        f.write(json.dumps(format_example(row)) + "\n")

with open("valid.jsonl", "w") as f:
    for _, row in valid.iterrows():
        f.write(json.dumps(format_example(row)) + "\n")

In [28]:
import json

def filter_jsonl(input_file, max_chars=16000):  # rough char limit ≈ 4096 tokens
    kept, removed = [], []
    with open(input_file) as f:
        for line in f:
            obj = json.loads(line)
            if len(obj["text"]) <= max_chars:
                kept.append(obj)
            else:
                removed.append(obj)
    
    with open(input_file, "w") as f:
        for obj in kept:
            f.write(json.dumps(obj) + "\n")
    
    print(f"Kept {len(kept)}, removed {len(removed)}")

filter_jsonl("train.jsonl")
filter_jsonl("valid.jsonl")

Kept 498, removed 37
Kept 55, removed 4


In [ ]:
!mlx_lm.lora \
  --model mlx-community/Qwen2.5-1.5B-Instruct-4bit \
  --train \
  --data ./ \
  --iters 500 \
  --batch-size 1 \
  --num-layers 16 \
  --learning-rate 1e-4 \
  --val-batches 5 \
  --save-every 100 \
  --max-seq-length 4096 \
  --adapter-path ./adapters

Loading pretrained model
Fetching 9 files: 100%|███████████████████████████| 9/9 [00:42<00:00,  4.69s/it]
Download complete: : 880MB [00:42, 20.9MB/s]              
Loading datasets
Training
Trainable parameters: 0.342% (5.276M/1543.714M)
Starting training..., iters: 500
Calculating loss...: 100%|████████████████████████| 5/5 [00:07<00:00,  1.44s/it]
Iter 1: Val loss 1.034, Val took 7.205s
Iter 10: Train loss 0.960, Learning Rate 1.000e-04, It/sec 0.714, Tokens/sec 534.457, Trained Tokens 7489, Peak mem 9.268 GB
[WARNING] Some sequences are longer than 4096 tokens. The longest sentence 4860 will be truncated to 4096. Consider pre-splitting your data to save memory.
Iter 20: Train loss 0.956, Learning Rate 1.000e-04, It/sec 0.224, Tokens/sec 332.054, Trained Tokens 22292, Peak mem 20.316 GB
Iter 30: Train loss 0.869, Learning Rate 1.000e-04, It/sec 0.505, Tokens/sec 545.096, Trained Tokens 33086, Peak mem 20.316 GB
Iter 40: Train loss 0.856, Learning Rate 1.000e-04, It/sec 0.523, Tokens

In [ ]:
from mlx_lm import load, generate

model, tokenizer = load(
    "mlx-community/Qwen2.5-0.5B-Instruct-4bit",
    adapter_path="./adapters"
)

prompt = "Penny has three times as many quarters as she has dimes. If the total value of her quarters and dimes is $4.25, find the total number of coins that Penny has."
messages = [{"role": "user", "content": prompt}]
prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
text = generate(model, tokenizer, prompt=prompt, max_tokens=1024, verbose=True)